# Chapter 13 - Sequences and Language

This notebook introduces a few core sequence ideas with a tiny in-notebook sentiment dataset. We will move from tokenization and padding to embeddings, then compare a simple mean-embedding classifier with a small GRU that reads variable-length sequences more explicitly.

The goal is intuition and runnable code, not a full NLP pipeline. A separate exercises notebook can extend these ideas later.

## Setup

We keep one small toy dataset and one shared batching function so each section can focus on a single idea.

In [ ]:
# !pip -q install torch matplotlib  # install dependencies if needed

import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader

plt.style.use("seaborn-v0_8")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_IDX = 0
UNK_IDX = 1

train_examples = [
    ("love this soft shirt", 1),
    ("great fit and soft fabric", 1),
    ("very comfortable and warm", 1),
    ("perfect size and nice color", 1),
    ("happy with this cozy sweater", 1),
    ("stylish look and great quality", 1),
    ("lightweight jacket feels great", 1),
    ("these shoes are very comfortable", 1),
    ("hate this rough shirt", 0),
    ("bad fit and itchy fabric", 0),
    ("very tight and uncomfortable", 0),
    ("cheap material and poor stitching", 0),
    ("returned it after one wear", 0),
    ("small size and rough seams", 0),
    ("the color faded quickly", 0),
    ("this jacket feels stiff", 0),
]

val_examples = [
    ("love the warm fabric", 1),
    ("stylish and comfortable hoodie", 1),
    ("rough seams and bad fit", 0),
    ("cheap shirt feels itchy", 0),
]

The three functions below form the NLP pipeline: `tokenize` splits a sentence into lowercase words, `build_vocab` maps each unique word to an integer index, and `encode` converts a sentence to a list of indices. These are the core concepts this chapter is introducing.

In [ ]:
def tokenize(text):
    return text.lower().split()


def build_vocab(examples):
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for text, _ in examples:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


vocab = build_vocab(train_examples)
id_to_token = {idx: token for token, idx in vocab.items()}


def encode(text):
    return [vocab.get(token, UNK_IDX) for token in tokenize(text)]

`collate_batch` pads each batch to the same length so sequences can be stacked into a tensor. The padding value (0) is ignored by the model — only real token positions contribute to the output.

In [ ]:
def collate_batch(batch):
    sequences = [torch.tensor(encode(text), dtype=torch.long) for text, _ in batch]
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor([label for _, label in batch], dtype=torch.long)
    return padded, lengths, labels


train_loader = DataLoader(
    train_examples,
    batch_size=4,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
    collate_fn=collate_batch,
)
val_loader = DataLoader(val_examples, batch_size=4, shuffle=False, collate_fn=collate_batch)

These three helpers handle training, evaluation, and prediction display. They are generic utilities — the interesting NLP work is in the pipeline above, not here.

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate_classifier(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for token_ids, lengths, labels in loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)
            logits = model(token_ids, lengths)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return total_loss / total_examples, total_correct / total_examples


def fit_classifier(model, train_loader, val_loader, *, epochs=30, lr=1e-2):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_acc": []}

    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for token_ids, lengths, labels in train_loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(token_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)

        _, val_acc = evaluate_classifier(model, val_loader)
        history["train_loss"].append(total_loss / total_examples)
        history["val_acc"].append(val_acc)

    return history


def show_predictions(model, examples):
    model.eval()
    padded, lengths, labels = collate_batch(examples)
    with torch.no_grad():
        probs = model(padded.to(device), lengths).softmax(dim=1)[:, 1].cpu()

    for (text, label), prob in zip(examples, probs):
        pred = int(prob >= 0.5)
        print(f"{text:<32} | true={label} pred={pred} p(pos)={prob.item():.2f}")


print(f"Training examples: {len(train_examples)}, validation examples: {len(val_examples)}")
print(f"Vocabulary size: {len(vocab)}")

## Tokenization and padding

A sequence model cannot read raw strings directly. We first split each sentence into tokens, map tokens to integer ids, and pad shorter sequences so a batch can share one rectangular tensor.

In [ ]:
demo_batch = train_examples[:3]
demo_tokens = [tokenize(text) for text, _ in demo_batch]
demo_ids, demo_lengths, demo_labels = collate_batch(demo_batch)

for (text, label), tokens, ids, length in zip(demo_batch, demo_tokens, demo_ids.tolist(), demo_lengths.tolist()):
    print(f"Text: {text}")
    print(f"Tokens: {tokens}")
    print(f"Token ids: {ids[:length]}")
    print(f"Length: {length}, label: {label}")
    print()

print(f"Padded batch shape: {tuple(demo_ids.shape)}")
print(f"Sequence lengths: {demo_lengths.tolist()}")
print(demo_ids)

## Embedding lookup

An embedding layer turns each token id into a learned dense vector. The output keeps the batch and sequence dimensions, then adds an embedding dimension at the end.

In [ ]:
embedding = nn.Embedding(len(vocab), 4, padding_idx=PAD_IDX)
embedded_demo = embedding(demo_ids)

print(f"Input token-id shape: {tuple(demo_ids.shape)}")
print(f"Embedding output shape: {tuple(embedded_demo.shape)}")
print(f"Padding vector norm: {embedding.weight[PAD_IDX].norm().item():.4f}")
print()

first_length = demo_lengths[0].item()
for token, token_id, vector in zip(
    demo_tokens[0],
    demo_ids[0, :first_length].tolist(),
    embedded_demo[0, :first_length].detach().numpy(),
):
    print(f"{token:<12} -> id {token_id:>2} -> {np.round(vector, 3)}")

## Tiny sentiment toy

A simple baseline is to embed each token, average the token vectors across the sentence, and classify from that mean embedding. This ignores word order, but it is often a useful first step.

In [ ]:
class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.classifier = nn.Linear(embed_dim, 2)

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        mask = (token_ids != PAD_IDX).unsqueeze(-1)
        summed = (embedded * mask).sum(dim=1)
        mean_embedding = summed / lengths.unsqueeze(1).to(embedded.dtype)
        return self.classifier(mean_embedding)


set_seed(42)
mean_model = MeanEmbeddingClassifier(len(vocab)).to(device)
mean_history = fit_classifier(mean_model, train_loader, val_loader, epochs=35, lr=0.03)
mean_val_loss, mean_val_acc = evaluate_classifier(mean_model, val_loader)

print(f"Mean-embedding validation accuracy: {mean_val_acc:.3f}")
show_predictions(mean_model, val_examples)

## GRU classifier

A GRU reads the sequence step by step, so order information can matter. We use `pack_padded_sequence` to tell the GRU which time steps are real tokens and which are only padding.

Mechanically, `pack_padded_sequence` sorts sequences by length and removes padding timesteps from the computation graph — the GRU only processes real tokens, which speeds up training and prevents padding from diluting the hidden state.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=12, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.gru(packed)
        return self.classifier(hidden[-1])


demo_inputs, demo_lengths, _ = next(iter(train_loader))
print(f"Batch token-id shape: {tuple(demo_inputs.shape)}")
print(f"Sequence lengths for packing: {demo_lengths.tolist()}")
print()

set_seed(42)
gru_model = GRUClassifier(len(vocab)).to(device)
gru_history = fit_classifier(gru_model, train_loader, val_loader, epochs=40, lr=0.02)
gru_val_loss, gru_val_acc = evaluate_classifier(gru_model, val_loader)

print(f"Mean-embedding validation accuracy: {mean_val_acc:.3f}")
print(f"GRU validation accuracy: {gru_val_acc:.3f}")
show_predictions(gru_model, val_examples)

## Embedding-space visualization

The mean-embedding model learned 3D token vectors, so we can inspect a tiny embedding space directly. On this toy problem, words with similar sentiment often drift toward similar parts of the space.

In [ ]:
sentiment_words = [
    "love", "soft", "great", "comfortable", "warm", "perfect",
    "hate", "rough", "itchy", "bad", "cheap", "tight",
]
word_points = torch.stack([
    mean_model.embedding.weight.detach().cpu()[vocab[word]]
    for word in sentiment_words
])
positive_words = {"love", "soft", "great", "comfortable", "warm", "perfect"}
colors = ["#55A868" if word in positive_words else "#C44E52" for word in sentiment_words]

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(projection="3d")
ax.scatter(word_points[:, 0], word_points[:, 1], word_points[:, 2], c=colors, s=70)

for word, point in zip(sentiment_words, word_points):
    ax.text(point[0].item(), point[1].item(), point[2].item(), word)

ax.set_title("Toy 3D embedding space")
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")
ax.set_zlabel("dim 3")
plt.show()

## Summary

This notebook kept Chapter 13 focused on a few sequence fundamentals:

- Tokenization maps text to tokens, then to integer ids.
- Padding makes variable-length sequences batchable, while sequence lengths keep track of the real content.
- Embeddings turn discrete tokens into learned dense vectors.
- A mean-embedding classifier is a simple baseline that ignores word order.
- A GRU can use packed sequences to read only the unpadded part of each sentence.
- Small learned embeddings can be inspected directly to build intuition about what the model has captured.

A separate exercises notebook can extend these ideas with more guided sequence comparisons.